Premier League Predictor

Importy:

In [12]:
from pathlib import Path

import pandas as pd
import numpy as np

In [13]:
df_history = pd.read_csv('../data/processed/PL_base_with_traits.csv')
df_history = df_history.dropna(subset=['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']).copy()

squad_values_path = Path('../data/raw/squad_values_by_season.csv')
value_features = []

if squad_values_path.exists():
    squad_values = pd.read_csv(squad_values_path)
    squad_values = squad_values[['SeasonFile', 'Team', 'SquadValueEUR']].drop_duplicates(['SeasonFile', 'Team'])

    home_values = squad_values.rename(columns={
        'Team': 'HomeTeam',
        'SquadValueEUR': 'HomeSquadValueEUR'
    })
    away_values = squad_values.rename(columns={
        'Team': 'AwayTeam',
        'SquadValueEUR': 'AwaySquadValueEUR'
    })

    df_history = df_history.merge(home_values, on=['SeasonFile', 'HomeTeam'], how='left')
    df_history = df_history.merge(away_values, on=['SeasonFile', 'AwayTeam'], how='left')

    if df_history[['HomeSquadValueEUR', 'AwaySquadValueEUR']].notna().all().all():
        df_history['SquadValueDiff'] = np.log(df_history['HomeSquadValueEUR']) - np.log(df_history['AwaySquadValueEUR'])
        value_features = ['SquadValueDiff']
    else:
        missing_values = df_history[df_history[['HomeSquadValueEUR', 'AwaySquadValueEUR']].isna().any(axis=1)]
        print(missing_values[['SeasonFile', 'HomeTeam', 'AwayTeam']].drop_duplicates().head(20).to_string(index=False))

base_diff_features = ['EloDiff', 'FormDiff', 'VenueFormDiff', 'H2HDiff']
diff_features = base_diff_features + value_features

base_cols = [
    'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'SeasonFile',
] + diff_features

df_home = df_history[base_cols].copy()
df_home = df_home.rename(columns={
    'HomeTeam': 'Team',
    'AwayTeam': 'Opponent',
    'FTHG': 'Goals',
    'FTAG': 'OpponentGoals'
})
df_home['IsHome'] = 1

df_away = df_history[base_cols].copy()
df_away = df_away.rename(columns={
    'AwayTeam': 'Team',
    'HomeTeam': 'Opponent',
    'FTAG': 'Goals',
    'FTHG': 'OpponentGoals'
})
df_away['IsHome'] = 0

for feature in diff_features:
    df_away[feature] = -df_away[feature]

df_poisson = pd.concat([df_home, df_away], ignore_index=True)

categorical_features = ['Team', 'Opponent']
numeric_features = ['IsHome'] + diff_features
poisson_features = categorical_features + numeric_features

print(f'Liczba meczow historycznych: {len(df_history)}')
print(f'Liczba wierszy dla modelu Poissona: {len(df_poisson)}')
print(f'Cechy liczbowe: {numeric_features}')
print(df_poisson[poisson_features + ['Goals']].head())

Liczba meczow historycznych: 4180
Liczba wierszy dla modelu Poissona: 8360
Cechy liczbowe: ['IsHome', 'EloDiff', 'FormDiff', 'VenueFormDiff', 'H2HDiff', 'SquadValueDiff']
          Team     Opponent  IsHome  EloDiff  FormDiff  VenueFormDiff  \
0  Bournemouth  Aston Villa       1      0.0       0.0            0.0   
1      Chelsea      Swansea       1      0.0       0.0            0.0   
2      Everton      Watford       1      0.0       0.0            0.0   
3    Leicester   Sunderland       1      0.0       0.0            0.0   
4   Man United    Tottenham       1      0.0       0.0            0.0   

   H2HDiff  SquadValueDiff  Goals  
0      0.0       -0.166091    0.0  
1      0.0        1.334026    2.0  
2      0.0        0.600880    2.0  
3      0.0        0.370245    4.0  
4      0.0        0.167943    1.0  


In [8]:
from sklearn.linear_model import PoissonRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

TEAM_NAME_MAP = {
    'AFC Bournemouth': 'Bournemouth',
    'Brighton & Hove Albion': 'Brighton',
    'Brighton and Hove Albion': 'Brighton',
    'Hull City': 'Hull',
    'Manchester City': 'Man City',
    'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle',
    'Nottingham Forest': "Nott'm Forest",
    'Tottenham Hotspur': 'Tottenham',
    'West Ham United': 'West Ham',
    'Wolverhampton Wanderers': 'Wolves',
    'Leeds United': 'Leeds',
    'Cardiff City': 'Cardiff'
}

future_season_file = 'season-2627.csv'
squad_value_lookup = {}
if 'squad_values' in globals():
    squad_value_lookup = {
        (row.SeasonFile, row.Team): float(row.SquadValueEUR)
        for row in squad_values.itertuples(index=False)
    }

def normalize_team_name(name):
    return TEAM_NAME_MAP.get(str(name).strip(), str(name).strip())

def normalize_fixture_names(fixtures):
    fixtures = fixtures.copy()
    fixtures['HomeTeam'] = fixtures['HomeTeam'].map(normalize_team_name)
    fixtures['AwayTeam'] = fixtures['AwayTeam'].map(normalize_team_name)
    return fixtures

X = df_poisson[poisson_features]
y = df_poisson['Goals'].astype(float)

preprocessor = ColumnTransformer(transformers=[
    ('teams', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('numbers', StandardScaler(), numeric_features)
])

poisson_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', PoissonRegressor(max_iter=1000, alpha=0.03))
])

poisson_model.fit(X, y)

last_home = df_history.groupby('HomeTeam').tail(1).set_index('HomeTeam')
last_away = df_history.groupby('AwayTeam').tail(1).set_index('AwayTeam')

def get_last_value(frame, team, column):
    if team in frame.index:
        return float(frame.at[team, column])
    return None

def latest_elo(team):
    team = normalize_team_name(team)
    values = []
    home_elo = get_last_value(last_home, team, 'HomeElo')
    away_elo = get_last_value(last_away, team, 'AwayElo')
    if home_elo is not None:
        values.append(home_elo)
    if away_elo is not None:
        values.append(away_elo)
    return float(np.mean(values)) if values else 1400.0

def latest_form(team, venue=None):
    team = normalize_team_name(team)
    if venue == 'home':
        value = get_last_value(last_home, team, 'HomeFormAtHome')
        if value is not None:
            return value
    if venue == 'away':
        value = get_last_value(last_away, team, 'AwayFormAtAway')
        if value is not None:
            return value

    values = []
    home_form = get_last_value(last_home, team, 'HomeForm')
    away_form = get_last_value(last_away, team, 'AwayForm')
    if home_form is not None:
        values.append(home_form)
    if away_form is not None:
        values.append(away_form)
    return float(np.mean(values)) if values else 1.0

def latest_squad_value(team, season_file=future_season_file):
    team = normalize_team_name(team)
    return squad_value_lookup.get((season_file, team))

def squad_value_diff(team, opponent, season_file=future_season_file):
    team_value = latest_squad_value(team, season_file)
    opponent_value = latest_squad_value(opponent, season_file)
    if team_value is None or opponent_value is None:
        return 0.0
    return float(np.log(team_value) - np.log(opponent_value))

h2h_history = {}
for row in df_history.itertuples(index=False):
    home = row.HomeTeam
    away = row.AwayTeam

    if row.FTHG > row.FTAG:
        home_points, away_points = 3, 0
    elif row.FTHG < row.FTAG:
        home_points, away_points = 0, 3
    else:
        home_points, away_points = 1, 1

    h2h_history.setdefault((home, away), []).append(home_points)
    h2h_history.setdefault((away, home), []).append(away_points)
    h2h_history[(home, away)] = h2h_history[(home, away)][-5:]
    h2h_history[(away, home)] = h2h_history[(away, home)][-5:]

def h2h_average(team, opponent):
    team = normalize_team_name(team)
    opponent = normalize_team_name(opponent)
    values = h2h_history.get((team, opponent), [])
    return float(np.mean(values)) if values else 1.0

def fixture_feature_row(team, opponent, is_home, venue_team, venue_opponent):
    feature_row = {
        'Team': team,
        'Opponent': opponent,
        'IsHome': is_home,
        'EloDiff': latest_elo(team) - latest_elo(opponent),
        'FormDiff': latest_form(team) - latest_form(opponent),
        'VenueFormDiff': latest_form(team, venue_team) - latest_form(opponent, venue_opponent),
        'H2HDiff': h2h_average(team, opponent) - h2h_average(opponent, team)
    }
    if 'SquadValueDiff' in numeric_features:
        feature_row['SquadValueDiff'] = squad_value_diff(team, opponent)
    return feature_row

def build_fixture_features(fixtures):
    fixtures = normalize_fixture_names(fixtures)
    home_features = []
    away_features = []

    for row in fixtures.itertuples(index=False):
        home = row.HomeTeam
        away = row.AwayTeam

        home_features.append(fixture_feature_row(home, away, 1, 'home', 'away'))
        away_features.append(fixture_feature_row(away, home, 0, 'away', 'home'))

    home_features = pd.DataFrame(home_features)[poisson_features]
    away_features = pd.DataFrame(away_features)[poisson_features]
    return fixtures, home_features, away_features

def predict_fixture_lambdas(fixtures):
    fixtures, home_features, away_features = build_fixture_features(fixtures)
    match_lambdas = fixtures[['HomeTeam', 'AwayTeam']].copy()
    match_lambdas['lambda_home'] = np.clip(poisson_model.predict(home_features), 0.05, 5.0)
    match_lambdas['lambda_away'] = np.clip(poisson_model.predict(away_features), 0.05, 5.0)
    return match_lambdas

print('Poisson model wytrenowany')
print(f'Srednia goli w danych: {y.mean():.2f}')

Poisson model wytrenowany
Srednia goli w danych: 1.41


In [14]:
import math


def make_poisson_rows(matches, diff_features):
    cols = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'SeasonFile'] + diff_features

    home_rows = matches[cols].copy()
    home_rows = home_rows.rename(columns={
        'HomeTeam': 'Team',
        'AwayTeam': 'Opponent',
        'FTHG': 'Goals',
        'FTAG': 'OpponentGoals'
    })
    home_rows['IsHome'] = 1

    away_rows = matches[cols].copy()
    away_rows = away_rows.rename(columns={
        'AwayTeam': 'Team',
        'HomeTeam': 'Opponent',
        'FTAG': 'Goals',
        'FTHG': 'OpponentGoals'
    })
    away_rows['IsHome'] = 0

    for feature in diff_features:
        away_rows[feature] = -away_rows[feature]

    model_rows = pd.concat([home_rows, away_rows], ignore_index=True)
    categorical_cols = ['Team', 'Opponent']
    numeric_cols = ['IsHome'] + diff_features
    model_features = categorical_cols + numeric_cols

    return model_rows, categorical_cols, numeric_cols, model_features


def make_match_features(matches, diff_features):
    home_features = matches[['HomeTeam', 'AwayTeam'] + diff_features].copy()
    home_features = home_features.rename(columns={
        'HomeTeam': 'Team',
        'AwayTeam': 'Opponent'
    })
    home_features['IsHome'] = 1

    away_features = matches[['AwayTeam', 'HomeTeam'] + diff_features].copy()
    away_features = away_features.rename(columns={
        'AwayTeam': 'Team',
        'HomeTeam': 'Opponent'
    })
    away_features['IsHome'] = 0

    for feature in diff_features:
        away_features[feature] = -away_features[feature]

    model_features = ['Team', 'Opponent', 'IsHome'] + diff_features
    return home_features[model_features], away_features[model_features]


def poisson_result_probs(lambda_home, lambda_away, max_goals=12):
    goals = np.arange(max_goals + 1)
    factorials = np.array([math.factorial(goal) for goal in goals])
    home_pmf = np.exp(-lambda_home) * np.power(lambda_home, goals) / factorials
    away_pmf = np.exp(-lambda_away) * np.power(lambda_away, goals) / factorials
    score_matrix = np.outer(home_pmf, away_pmf)
    score_matrix = score_matrix / score_matrix.sum()
    home_win = np.tril(score_matrix, -1).sum()
    draw = np.trace(score_matrix)
    away_win = np.triu(score_matrix, 1).sum()
    return home_win, draw, away_win


def evaluate_poisson_features(diff_features, label, min_train_seasons=4):
    seasons = sorted(df_history['SeasonFile'].unique())
    rows = []

    for test_season in seasons[min_train_seasons:]:
        train_matches = df_history[df_history['SeasonFile'] < test_season]
        test_matches = df_history[df_history['SeasonFile'] == test_season]

        train_rows, categorical_cols, numeric_cols, model_features = make_poisson_rows(train_matches, diff_features)

        preprocessor_eval = ColumnTransformer(transformers=[
            ('teams', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
            ('numbers', StandardScaler(), numeric_cols)
        ])

        model_eval = Pipeline(steps=[
            ('preprocessor', preprocessor_eval),
            ('model', PoissonRegressor(max_iter=1000, alpha=0.03))
        ])

        model_eval.fit(train_rows[model_features], train_rows['Goals'].astype(float))

        home_features, away_features = make_match_features(test_matches, diff_features)
        lambda_home = np.clip(model_eval.predict(home_features), 0.05, 5.0)
        lambda_away = np.clip(model_eval.predict(away_features), 0.05, 5.0)

        actual_home_goals = test_matches['FTHG'].astype(float).to_numpy()
        actual_away_goals = test_matches['FTAG'].astype(float).to_numpy()
        goal_errors = np.concatenate([
            actual_home_goals - lambda_home,
            actual_away_goals - lambda_away
        ])

        actual_results = np.where(
            actual_home_goals > actual_away_goals,
            'H',
            np.where(actual_home_goals == actual_away_goals, 'D', 'A')
        )

        predicted_results = []
        actual_probs = []

        for actual_result, lh, la in zip(actual_results, lambda_home, lambda_away):
            home_prob, draw_prob, away_prob = poisson_result_probs(lh, la)
            probs = {'H': home_prob, 'D': draw_prob, 'A': away_prob}
            predicted_results.append(max(probs, key=probs.get))
            actual_probs.append(max(probs[actual_result], 1e-12))

        rows.append({
            'Model': label,
            'Season': test_season.replace('season-', '').replace('.csv', ''),
            'MAE_goals': np.mean(np.abs(goal_errors)),
            'RMSE_goals': np.sqrt(np.mean(goal_errors ** 2)),
            'Acc_1X2_%': np.mean(np.array(predicted_results) == actual_results) * 100,
            'LogLoss_1X2': -np.mean(np.log(actual_probs))
        })

    return pd.DataFrame(rows)


feature_sets = {
    'teams_only': [],
    'current': ['EloDiff', 'FormDiff', 'VenueFormDiff', 'H2HDiff']
}

if 'SquadValueDiff' in df_history.columns:
    feature_sets['current_plus_squad_value'] = ['EloDiff', 'FormDiff', 'VenueFormDiff', 'H2HDiff', 'SquadValueDiff']

backtest_results = pd.concat(
    [evaluate_poisson_features(features, name) for name, features in feature_sets.items()],
    ignore_index=True
)

backtest_summary = backtest_results.groupby('Model', as_index=False).agg({
    'MAE_goals': 'mean',
    'RMSE_goals': 'mean',
    'Acc_1X2_%': 'mean',
    'LogLoss_1X2': 'mean'
})

print(backtest_summary.round(3).to_string(index=False))
print()
print(backtest_results.round(3).to_string(index=False))

                   Model  MAE_goals  RMSE_goals  Acc_1X2_%  LogLoss_1X2
                 current      0.930       1.190     52.782        0.985
current_plus_squad_value      0.926       1.185     52.932        0.981
              teams_only      0.947       1.209     50.301        1.013

                   Model Season  MAE_goals  RMSE_goals  Acc_1X2_%  LogLoss_1X2
              teams_only   1920      0.885       1.137     51.842        0.985
              teams_only   2021      0.973       1.254     50.000        1.033
              teams_only   2122      0.940       1.197     53.947        0.973
              teams_only   2223      0.987       1.270     51.316        1.015
              teams_only   2324      0.979       1.293     55.789        0.975
              teams_only   2425      0.956       1.217     42.368        1.061
              teams_only   2526      0.908       1.097     46.842        1.051
                 current   1920      0.897       1.142     53.947        0.978


In [15]:
sanity_fixtures = pd.DataFrame([
    {'HomeTeam': 'Liverpool', 'AwayTeam': 'Sunderland'},
    {'HomeTeam': 'Burnley', 'AwayTeam': 'Arsenal'},
    {'HomeTeam': 'Man City', 'AwayTeam': 'Leeds United'},
    {'HomeTeam': 'Everton', 'AwayTeam': 'Fulham'},
    {'HomeTeam': 'Tottenham', 'AwayTeam': 'Chelsea'}
])

sanity_df = predict_fixture_lambdas(sanity_fixtures)
sanity_df['expected_goals_total'] = sanity_df['lambda_home'] + sanity_df['lambda_away']
sanity_df[['lambda_home', 'lambda_away', 'expected_goals_total']] = sanity_df[['lambda_home', 'lambda_away', 'expected_goals_total']].round(2)
sanity_df['Mecz'] = sanity_df['HomeTeam'] + ' - ' + sanity_df['AwayTeam']

print(sanity_df[['Mecz', 'lambda_home', 'lambda_away', 'expected_goals_total']].to_string(index=False))

                  Mecz  lambda_home  lambda_away  expected_goals_total
Liverpool - Sunderland         2.17         0.85                  3.03
     Burnley - Arsenal         1.06         1.55                  2.61
      Man City - Leeds         2.72         0.78                  3.50
      Everton - Fulham         1.43         1.12                  2.55
   Tottenham - Chelsea         1.43         1.41                  2.84


In [22]:
N_SIMS = 1500
MIN_TEAM_UNCERTAINTY_SIGMA = 0.02
MAX_TEAM_UNCERTAINTY_SIGMA = 0.12
MAX_LOW_HISTORY_SHRINKAGE = 0.50
RECENT_HISTORY_SEASONS = 3
HISTORY_SCALE_MATCHES = 76
OPPONENT_NOISE_WEIGHT = 0.35
MAX_STRENGTH_NOISE = 0.18
rng = np.random.default_rng(42)

df_future = pd.read_csv('../data/raw/pl_seasons/season-2627.csv')
df_future = normalize_fixture_names(df_future)

match_lambdas = predict_fixture_lambdas(df_future)
teams = sorted(set(match_lambdas['HomeTeam']) | set(match_lambdas['AwayTeam']))
recent_seasons = sorted(df_history['SeasonFile'].unique())[-RECENT_HISTORY_SEASONS:]
recent_history = df_history[df_history['SeasonFile'].isin(recent_seasons)]
home_goal_mean = float(df_poisson.loc[df_poisson['IsHome'] == 1, 'Goals'].mean())
away_goal_mean = float(df_poisson.loc[df_poisson['IsHome'] == 0, 'Goals'].mean())

recent_match_count = {
    team: int(((recent_history['HomeTeam'] == team) | (recent_history['AwayTeam'] == team)).sum())
    for team in teams
}
team_uncertainty_sigma = {
    team: MIN_TEAM_UNCERTAINTY_SIGMA + (MAX_TEAM_UNCERTAINTY_SIGMA - MIN_TEAM_UNCERTAINTY_SIGMA) * np.exp(-recent_match_count[team] / HISTORY_SCALE_MATCHES)
    for team in teams
}
team_shrinkage = {
    team: MAX_LOW_HISTORY_SHRINKAGE * np.exp(-recent_match_count[team] / HISTORY_SCALE_MATCHES)
    for team in teams
}

adjusted_match_lambdas = match_lambdas.copy()
adjusted_home_lambdas = []
adjusted_away_lambdas = []

for row in match_lambdas.itertuples(index=False):
    home_shrinkage = team_shrinkage[row.HomeTeam]
    away_shrinkage = team_shrinkage[row.AwayTeam]
    adjusted_home_lambdas.append((1 - home_shrinkage) * row.lambda_home + home_shrinkage * home_goal_mean)
    adjusted_away_lambdas.append((1 - away_shrinkage) * row.lambda_away + away_shrinkage * away_goal_mean)

adjusted_match_lambdas['lambda_home'] = adjusted_home_lambdas
adjusted_match_lambdas['lambda_away'] = adjusted_away_lambdas

positions_count = {team: np.zeros(len(teams), dtype=int) for team in teams}
total_points = {team: 0 for team in teams}
total_gf = {team: 0 for team in teams}
total_ga = {team: 0 for team in teams}

for _ in range(N_SIMS):
    table = {
        team: {'Pts': 0, 'GF': 0, 'GA': 0} for team in teams
    }
    strength_noise = {
        team: float(np.clip(rng.normal(0, team_uncertainty_sigma[team]), -MAX_STRENGTH_NOISE, MAX_STRENGTH_NOISE))
        for team in teams
    }

    for row in adjusted_match_lambdas.itertuples(index=False):
        home = row.HomeTeam
        away = row.AwayTeam

        lambda_home = row.lambda_home * np.exp(strength_noise[home] - OPPONENT_NOISE_WEIGHT * strength_noise[away])
        lambda_away = row.lambda_away * np.exp(strength_noise[away] - OPPONENT_NOISE_WEIGHT * strength_noise[home])
        lambda_home = float(np.clip(lambda_home, 0.05, 5.0))
        lambda_away = float(np.clip(lambda_away, 0.05, 5.0))

        home_goals = int(rng.poisson(lambda_home))
        away_goals = int(rng.poisson(lambda_away))

        table[home]['GF'] += home_goals
        table[home]['GA'] += away_goals
        table[away]['GF'] += away_goals
        table[away]['GA'] += home_goals

        if home_goals > away_goals:
            table[home]['Pts'] += 3
        elif away_goals > home_goals:
            table[away]['Pts'] += 3
        else:
            table[home]['Pts'] += 1
            table[away]['Pts'] += 1

    sorted_teams = sorted(
        teams,
        key=lambda team: (
            table[team]['Pts'],
            table[team]['GF'] - table[team]['GA'],
            table[team]['GF']
        ),
        reverse=True
    )

    for rank, team in enumerate(sorted_teams):
        positions_count[team][rank] += 1

    for team in teams:
        total_points[team] += table[team]['Pts']
        total_gf[team] += table[team]['GF']
        total_ga[team] += table[team]['GA']

results = []
for team in teams:
    chances = positions_count[team] / N_SIMS
    xgf = total_gf[team] / N_SIMS
    xga = total_ga[team] / N_SIMS

    results.append({
        'Druzyna': team,
        'xPts': round(total_points[team] / N_SIMS, 1),
        'xGF': round(xgf, 1),
        'xGA': round(xga, 1),
        'xGD': round(xgf - xga, 1),
        'Tytul (%)': round(chances[0] * 100, 1),
        'Top 4 (%)': round(np.sum(chances[:4]) * 100, 1),
        'Spadek (%)': round(np.sum(chances[-3:]) * 100, 1),
        'Niepewnosc': round(team_uncertainty_sigma[team], 2),
        'Regresja': round(team_shrinkage[team], 2)
    })

final_poisson_table = pd.DataFrame(results).sort_values(
    by=['xPts', 'xGD', 'xGF'],
    ascending=False
).reset_index(drop=True)
final_poisson_table.index += 1

print('\nPrzewidywana tabela Premier League 2026/2027 - Poisson Monte Carlo')
print(f'Recent history seasons: {recent_seasons}')
print('-' * 105)
print(final_poisson_table.to_string())


Przewidywana tabela Premier League 2026/2027 - Poisson Monte Carlo
Recent history seasons: ['season-2324.csv', 'season-2425.csv', 'season-2526.csv']
---------------------------------------------------------------------------------------------------------
           Druzyna  xPts   xGF   xGA   xGD  Tytul (%)  Top 4 (%)  Spadek (%)  Niepewnosc  Regresja
1         Man City  78.2  79.9  38.3  41.7       59.5       96.8         0.0        0.04      0.11
2          Arsenal  73.0  72.5  40.2  32.4       28.3       87.9         0.0        0.04      0.11
3        Liverpool  65.1  65.6  45.9  19.7        5.9       57.3         0.1        0.04      0.11
4          Chelsea  60.5  59.5  47.4  12.0        1.8       33.1         0.4        0.04      0.11
5       Man United  58.8  57.5  47.8   9.8        0.9       26.3         0.8        0.04      0.11
6         Brighton  56.6  55.1  48.7   6.4        0.7       19.3         1.5        0.04      0.11
7      Bournemouth  55.2  56.6  52.9   3.6        0

In [21]:
import subprocess
import sys
from pathlib import Path

forecast_script = Path('run_forecast.py')
if not forecast_script.exists():
    forecast_script = Path('src/run_forecast.py')

subprocess.run([sys.executable, str(forecast_script)], check=True)

CompletedProcess(args=['c:\\Users\\maksy\\pl_predictor\\venv\\Scripts\\python.exe', 'run_forecast.py'], returncode=0)